# 02 · Wordle — multi-turn **agentic** GRPO + play-the-game evaluation

Notebook 01's env was single-turn (one question → one answer). **Wordle is genuinely
agentic**: up to **6 guesses** per episode, and each guess returns colour-coded feedback
(GREEN/YELLOW/GRAY) the model must *reason over* to choose its next guess. The episode is
**stateful** and **multi-turn** — the closest OpenEnv shape to a real tool-using agent.

The training mechanics are identical to notebook 01 (`environment_factory` + GRPO), which
is the point: once you have the pattern, swapping the environment is the only change. We
reuse the same SFT→GRPO→evaluate spine and highlight what differs for a multi-turn env.

> **Why Wordle and not a "coding env"?** OpenEnv's coding agent (`opencode_env`) is real
> but needs an **E2B sandbox** *and* an OpenAI-compatible LLM endpoint — two external
> dependencies that break "runs anywhere." Wordle (`textarena`) is multi-turn, agentic,
> needs **no API key**, and is the canonical agentic tutorial. The production coding path
> is documented in the README for when those dependencies are available.

> **Runtime targets — this standalone notebook runs on HF Jobs or Colab.** It self-installs
> its dependencies, authenticates with `notebook_login()` (or `HF_TOKEN`), connects
> to an environment by **URL** (a variable you can repoint to your own Space), and
> auto-detects the GPU with a model-size fallback. There are **no hard-coded paths**.

<!-- @inject:run-recipes -->
## ▶ How to run this notebook (HF Jobs · Google Colab)

This notebook is standalone. Pick a lane. (Dependencies and the env
URL below are taken from this notebook's own cells.)

### A · HF Jobs — run the notebook on Hugging Face infra

`hf jobs run` is *docker-run for HF infra*: the compute happens on HF's cloud, you only
submit. The control plane is plain HTTPS, Requires pre-paid credits on your HF account.

```bash
# one-time: hf auth login   (or rely on a local HF_TOKEN)
hf jobs run \
  --flavor a10g-small \
  --timeout 10800 \
  --secrets HF_TOKEN \
  -e SMOKE=0 -e REPORT_TO=trackio \
  -e ENV_BASE_URL=https://openenv-wordle.hf.space \
  python:3.12 \
  bash -c "pip install -q papermill && \
           pip install -q trl openenv "transformers>=5.3.0" trackio jmespath nest_asyncio datasets && \
           pip install -q --no-deps git+https://huggingface.co/spaces/openenv/wordle && \
           papermill 02_wordle_agentic_grpo.ipynb out.ipynb"
```

Notes: `--secrets HF_TOKEN` forwards your local token (needed if the notebook pushes to the
Hub). `--flavor` options: `cpu-basic`, `t4-small`, `a10g-small`, `a100-large`, … (`hf jobs`
docs). Add `--detach` to background it; watch with `hf jobs logs <id>`.

### B · HF Jobs — full run with vLLM (faster rollouts)

Same as A but on a bigger GPU with vLLM enabled (`USE_VLLM=1`). vLLM works here because an
HF Job is **not** an IPython runtime (the reason it stays OFF in-notebook).

```bash
hf jobs run \
  --flavor a100-large \
  --timeout 21600 \
  --secrets HF_TOKEN \
  -e SMOKE=0 -e USE_VLLM=1 -e REPORT_TO=trackio \
  -e ENV_BASE_URL=https://openenv-wordle.hf.space \
  python:3.12 \
  bash -c "pip install -q papermill vllm && \
           pip install -q trl openenv "transformers>=5.3.0" trackio jmespath nest_asyncio datasets && \
           pip install -q --no-deps git+https://huggingface.co/spaces/openenv/wordle && \
           papermill 02_wordle_agentic_grpo.ipynb out.ipynb"
```

### C · Google Colab — independent run

Open Colab → set runtime to **GPU** → in the first cell:

```python
# Colab cell 1 — auth + knobs, then run the rest of the notebook top-to-bottom.
import os
os.environ["SMOKE"] = "0"                 # full run (omit for a quick smoke)
os.environ["REPORT_TO"] = "trackio"       # live charts; or "none" for headless
os.environ["ENV_BASE_URL"] = "https://openenv-wordle.hf.space"
from huggingface_hub import notebook_login
notebook_login()                          # paste a token with write access
```

Then **Runtime → Run all**. Colab has open egress, so the hosted env Space works directly
(no in-job uvicorn needed). The install cell at the top of the notebook handles
dependencies.

For a longer run on an A100 Colab, set `os.environ["USE_VLLM"] = "1"` *before* the GRPO config cell — Colab's kernel tolerates vLLM colocate on a single A100.

---


## 0 · Install

The Wordle env client ships from the `openenv/wordle` Space. `trl[vllm]` is the
tutorial default; in-notebook we keep vLLM off (IPython init issue) and enable it only
via `USE_VLLM=1` on an HF Job.

In [ ]:
%pip install -q trl openenv "transformers>=5.3.0" trackio jmespath nest_asyncio datasets
%pip install -q --no-deps git+https://huggingface.co/spaces/openenv/wordle

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
# --- Authenticate with the Hugging Face Hub (portable) -------------------------
# Prefers an already-set HF_TOKEN (HF Jobs / Colab secret); falls back to the
# interactive widget. Never hard-codes a token.
import os

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login

    login(token=os.environ["HF_TOKEN"])
    print("Authenticated via HF_TOKEN.")
else:
    try:
        from huggingface_hub import notebook_login

        notebook_login()
    except Exception as exc:  # non-interactive without a token
        print(f"notebook_login unavailable ({exc}); set HF_TOKEN before running.")

In [ ]:
# Resolve your Hub username automatically — no hard-coded usernames downstream.
from huggingface_hub import whoami

try:
    HF_USERNAME = whoami()["name"]
    print(f"Hub user: {HF_USERNAME}")
except Exception as exc:
    HF_USERNAME = os.environ.get("HF_USERNAME", "")
    print(f"Could not resolve whoami() ({exc}); using HF_USERNAME={HF_USERNAME!r}.")

In [ ]:
# --- Auto-detect compute + pick a model that fits ------------------------------
# Larger model only when there is VRAM headroom; otherwise fall back. You can
# override by setting MODEL_NAME in the environment before launching.
import torch

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    DEVICE = "cuda"
else:
    vram_gb = 0.0
    DEVICE = "cpu"

DEFAULT_MODEL = "Qwen/Qwen3-1.7B" if vram_gb >= 24 else "Qwen/Qwen3-0.6B"
MODEL_NAME = os.environ.get("MODEL_NAME", DEFAULT_MODEL)
print(f"device={DEVICE}  vram={vram_gb:.1f} GB  ->  MODEL_NAME={MODEL_NAME}")

In [ ]:
import os

SMOKE = os.environ.get("SMOKE", "1") not in ("0", "false", "False")
ENV_BASE_URL = os.environ.get("ENV_BASE_URL", "https://openenv-wordle.hf.space")
GRPO_MAX_STEPS = 5 if SMOKE else 150
N_EVAL_GAMES = 3 if SMOKE else 20
print(f"SMOKE={SMOKE}  grpo_steps={GRPO_MAX_STEPS}  eval_games={N_EVAL_GAMES}  env={ENV_BASE_URL}")

## 1 · System prompt — teach the rules and the tool

The prompt states the Wordle rules, the feedback colour code, and — critically — that
the model must call the `guess` tool. The `environment_factory` loop drives tool calls,
so the model has to know the tool exists.

In [ ]:
WORDLE_PROMPT = """You are an expert Wordle solver with deep knowledge of English vocabulary, letter frequency patterns, and optimal guessing strategies.

Follow these rules to play Wordle:

1. The target is a 5-letter English word
2. You have 6 attempts to guess the correct word
3. After each guess, you receive color-coded feedback:
   - GREEN (G): Letter is correct and in the correct position
   - YELLOW (Y): Letter is in the word but in the wrong position
   - GRAY (X): Letter is not in the word at all
4. All guesses must be valid 5-letter English words
5. You cannot reuse a word you've already guessed
6. Use the tool `guess` to make a guess.
"""

## 2 · The multi-turn environment class

Two things differ from notebook 01's single-turn env:

- **`reset()` returns the running feedback transcript**, and the env appends new feedback
  each turn. We keep `self._last_full_feedback` and slice out only the *newly appended*
  part so the model sees just the latest result.
- **`done` is set by the env**, not by us — the game ends on a win or after 6 guesses, so
  a single rollout spans multiple `guess` tool calls. The trainer keeps stepping until
  `done=True`.

`guess` is the one tool (public + docstring → auto-discovered). It penalises invalid
moves (reward 0) and otherwise records the env's reward.

In [ ]:
from textarena_env import TextArenaAction, TextArenaEnv


class WordleEnv:
    """Multi-turn Wordle rollout: up to 6 `guess` tool calls until done."""

    def __init__(self):
        self.client = TextArenaEnv(base_url=ENV_BASE_URL).sync()

    def reset(self, **kwargs) -> None | str:
        result = self.client.reset()
        # Env returns cumulative feedback; store the full text so we can diff each turn.
        self._last_full_feedback = result.observation.messages[0].content
        self.reward = 0.0
        self.done = False
        return self._last_full_feedback

    def guess(self, guess: str) -> str:
        """Make a guess in the Wordle environment.

        Args:
            guess: The guessed word, formatted as '[abcde]'.

        Returns:
            The feedback message from the environment.
        """
        if self.done:
            raise ValueError("Game over.")
        result = self.client.step(TextArenaAction(message=guess))
        full = result.observation.messages[0].content
        feedback = full[len(self._last_full_feedback):]  # only the newly appended part
        self._last_full_feedback = full
        self.reward = 0.0 if "You attempted an invalid move" in feedback else result.reward
        self.done = result.done
        return feedback


def reward_func(environments, **kwargs) -> list[float]:
    return [env.reward for env in environments]

## 3 · Dataset + GRPO config

Same dummy-prompt trick (count = episodes). The notable config difference vs. notebook
01 is a larger `max_completion_length` (multi-turn games are longer) and a bigger
`gradient_accumulation_steps` per the Wordle tutorial.

In [ ]:
from datasets import Dataset

N_PROMPTS = 30 if SMOKE else 3000
dataset = Dataset.from_dict(
    {"prompt": [[{"role": "user", "content": WORDLE_PROMPT}] for _ in range(N_PROMPTS)]}
)

In [ ]:
import re as _re

from trl import GRPOConfig

# Hyphen-lowercase slug for clean HF/trackio Space ids.
GRPO_OUT = _re.sub(r"-+", "-", _re.sub(r"[^a-z0-9]+", "-", f"wordle-grpo-{MODEL_NAME.split('/')[-1]}".lower())).strip("-")

# Logging backend knob: "trackio" (default) or "none" for a fully offline/headless run.
REPORT_TO = os.environ.get("REPORT_TO", "trackio")
_report_kwargs = {"report_to": REPORT_TO}
if REPORT_TO == "trackio":
    _report_kwargs["trackio_space_id"] = GRPO_OUT

grpo_config = GRPOConfig(
    num_train_epochs=1,
    max_steps=GRPO_MAX_STEPS,
    learning_rate=1e-6,
    gradient_accumulation_steps=8 if SMOKE else 64,
    per_device_train_batch_size=1,
    warmup_steps=min(10, GRPO_MAX_STEPS),
    optim="adamw_torch",
    max_grad_norm=1.0,
    num_generations=2,
    max_completion_length=1024,           # multi-turn games are longer than chain_sum
    log_completions=True,
    num_completions_to_print=2,
    chat_template_kwargs={"enable_thinking": False},
    output_dir=GRPO_OUT,
    # Push weights to a repo DISTINCT from the Trackio Space id (=GRPO_OUT),
    # otherwise push_to_hub sees the Trackio-owned repo and skips as 'no files
    # modified'. A separate -model repo gets the actual checkpoint commit.
    hub_model_id=f"{HF_USERNAME}/{GRPO_OUT}-model" if HF_USERNAME else f"{GRPO_OUT}-model",
    logging_steps=1 if SMOKE else 10,
    save_strategy="no",
    gradient_checkpointing=True,
    **_report_kwargs,
    push_to_hub=not SMOKE,
    # vLLM OFF in-notebook (IPython init). The optional cell below enables it for an HF Job.
)

In [ ]:
# @inject:vllm-optional
# --- OPTIONAL: vLLM-accelerated GRPO rollouts (5-10x faster generation) ---------
# OFF by default: vLLM's init breaks under IPython/Jupyter, so leave USE_VLLM=0 for an
# in-notebook run. Enable it (USE_VLLM=1) ONLY in a non-IPython context: an HF Job, a
# or a fresh Colab runtime executed as a script. Colocate mode shares
# the single training GPU (right for Colab / one-GPU HF-Job flavors).
#   Verified TRL v1.7.0 params: use_vllm, vllm_mode="colocate"|"server",
#   vllm_gpu_memory_utilization. (server mode is multi-GPU; not used here.)
import os

USE_VLLM = os.environ.get("USE_VLLM", "0") not in ("0", "false", "False")
if USE_VLLM:
    grpo_config.use_vllm = True
    grpo_config.vllm_mode = "colocate"
    # Leave room for the training copy of the model in colocate mode; tune per GPU/model.
    grpo_config.vllm_gpu_memory_utilization = float(
        os.environ.get("VLLM_GPU_MEM_UTIL", "0.3")
    )
    print(
        f"[vLLM] enabled: mode=colocate gpu_mem_util={grpo_config.vllm_gpu_memory_utilization} "
        "(requires a non-IPython runtime + `pip install vllm`)"
    )
else:
    print("[vLLM] disabled (USE_VLLM=0). In-notebook generation uses HF generate().")

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=reward_func,
    train_dataset=dataset,
    args=grpo_config,
    environment_factory=WordleEnv,
)
trainer.train()
trainer.save_model(GRPO_OUT)
if not SMOKE:
    trainer.push_to_hub(commit_message="GRPO fine-tune on Wordle (textarena)")

## 4 · Training signal — reward delta

Same quick check as notebook 01: mean reward, first-5 vs last-5 logged steps.

In [ ]:
import statistics

rewards = [log["reward"] for log in trainer.state.log_history if "reward" in log]
if len(rewards) < 5:
    print(f"Only {len(rewards)} reward logs — increase max_steps for a clean delta.")
else:
    initial, final = statistics.mean(rewards[:5]), statistics.mean(rewards[-5:])
    print(f"initial={initial:.2%}  final={final:.2%}  delta={(final - initial) * 100:+.2f}pp")

## 5 · Test the trained agent — play full games

For a multi-turn agent, the meaningful test is **playing complete games**: load the
trained model, let it guess turn-by-turn against live feedback, and measure the win rate
over `N_EVAL_GAMES`. This is the canonical `play_wordle` harness, wrapped to score a win
(`env.done and env.reward > 0`) and report aggregate performance — the agentic analogue
of notebook 01's accuracy.

In [ ]:
import json
import re

from transformers import AutoModelForCausalLM, AutoTokenizer


def play_one_game(model, tokenizer, verbose=False):
    """Play a single Wordle game with the model; return (won: bool, final_reward: float)."""
    env = WordleEnv()
    try:
        obs = env.reset()
        messages = [{"role": "user", "content": WORDLE_PROMPT}]
        if obs:
            messages.append({"role": "user", "content": obs})

        for _turn in range(6):
            if env.done:
                break
            prompt_text = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False, enable_thinking=False
            )
            inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)
            out = model.generate(**inputs, max_new_tokens=512)
            text = tokenizer.decode(out[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
            if verbose:
                print(f"  model: {text[:120]}")
            try:
                if "guess" in text and "{" in text:
                    args = json.loads(text[text.index("{"):text.rindex("}") + 1])
                    args = args.get("arguments", args)
                    word = args.get("guess", "")
                else:
                    m = re.search(r"\[([a-zA-Z]{5})\]", text)
                    word = m.group(1) if m else text.strip()[:5]
                feedback = env.guess(f"[{word}]")
                messages.append({"role": "assistant", "content": text})
                messages.append({"role": "user", "content": feedback})
            except Exception as exc:  # malformed output ends the game
                if verbose:
                    print(f"  parse error: {exc}")
                break

        return (env.done and env.reward > 0), float(env.reward)
    finally:
        # Release this game's env session so it doesn't leak against the server's
        # max_concurrent_envs cap across N_EVAL_GAMES (mirrors notebook 01's eval close).
        try:
            env.client.close()
        except Exception:
            pass


In [ ]:
# Win-rate of the trained agent over several games.
fine_tuned = AutoModelForCausalLM.from_pretrained(GRPO_OUT, torch_dtype="auto", device_map="auto")
ft_tokenizer = AutoTokenizer.from_pretrained(GRPO_OUT)

wins, total_reward = 0, 0.0
for g in range(N_EVAL_GAMES):
    won, r = play_one_game(fine_tuned, ft_tokenizer, verbose=(g == 0))
    wins += int(won)
    total_reward += r
    print(f"game {g + 1}/{N_EVAL_GAMES}: {'WON' if won else 'lost'}  reward={r:.2f}")

print(f"\nwin rate: {wins}/{N_EVAL_GAMES} = {wins / N_EVAL_GAMES:.0%}   mean reward={total_reward / N_EVAL_GAMES:.2f}")

## Recap

Same `environment_factory` + GRPO spine as notebook 01, applied to a **multi-turn agentic**
environment. The only real changes were the env class (stateful feedback, env-driven
`done`) and a longer completion budget. The trained agent is judged by **playing full
games** and measuring win rate — the agentic counterpart to held-out accuracy.

This is the template for any OpenEnv environment: define the env wrapper (tools = public
documented methods), a reward function that reads `env.reward`, a dummy prompt dataset for
the episode count, and hand the class to `GRPOTrainer(environment_factory=...)`. Swap the
env, keep the spine.